# 06. CNN Integrated Gradients

Integrated Gradients는 `<pad>` baseline에서 실제 입력 embedding까지의 경로를 따라 gradient를 누적한 attribution이다. 이 노트북은 `cnn_integrated_gradients.csv`를 읽어 token별 IG 중요도를 확인하고 Gradient x Input과 비교한다.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
if (cwd / 'xai_outputs').exists(): OUT = cwd / 'xai_outputs'
elif (cwd.parent / 'xai_outputs').exists(): OUT = cwd.parent / 'xai_outputs'
else: OUT = Path('XAI/CNN/xai_outputs')

selected = pd.read_csv(OUT / 'cnn_xai_selected_samples.csv')
ig = pd.read_csv(OUT / 'cnn_integrated_gradients.csv')
grad = pd.read_csv(OUT / 'cnn_gradient_x_input.csv')
print(OUT.resolve(), len(ig))

In [ ]:
sample_id = 'custom_3' if 'custom_3' in set(selected['sample_id']) else selected['sample_id'].iloc[0]
sample = selected[selected['sample_id'] == sample_id].iloc[0]
rows = ig[ig['sample_id'] == sample_id].sort_values('normalized_abs_score', ascending=False)

print('sample_id:', sample_id)
print('text:', sample['text'])
print('pred:', sample['pred_label_name'], 'target:', sample['target_class_name'])
display(rows[['position', 'token', 'signed_score', 'abs_score', 'normalized_abs_score', 'ig_steps', 'baseline']].head(15))

In [ ]:
g = grad[grad['sample_id'] == sample_id][['position', 'token', 'normalized_abs_score']].rename(columns={'normalized_abs_score': 'gradient_x_input'})
i = ig[ig['sample_id'] == sample_id][['position', 'normalized_abs_score']].rename(columns={'normalized_abs_score': 'integrated_gradients'})
merged = g.merge(i, on='position', how='inner').sort_values('position')
display(merged)

ax = merged.plot(x='token', y=['gradient_x_input', 'integrated_gradients'], kind='bar', figsize=(10, 4))
ax.set_title(f'Gradient x Input vs Integrated Gradients: {sample_id}')
ax.set_xlabel('token')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()